# Gold Layer — NSW Air Quality

Pre-aggregated tables built for questions rather than for storage.
Dashboards query these, never silver.

| Table | Grain — what one row represents |
|---|---|
| `daily_site_summary` | One site, one date, one parameter |
| `site_hourly_profile` | One site, one parameter, one hour-of-day |
| `regional_trend` | One region, one month, one parameter |
| `exceedance_events` | One site, one date, one parameter (exceedances only) |

Every grain is proven at the end of this notebook.

In [0]:
from pyspark.sql import functions as F

## 1. Daily site summary

**Question it answers:** what was the air like at each station each day?

This is the workhorse table — most dashboard charts read from it.

`completeness_pct` matters: a daily average built from 3 valid hours is not
comparable to one built from 24. Carrying completeness alongside the average lets
you filter later, and lets you say honestly that you tracked it.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.daily_site_summary AS
SELECT
  f.site_id,
  s.site_name,
  s.region,
  s.latitude,
  s.longitude,
  f.obs_date,
  f.parameter_code,
  f.units,
  ROUND(AVG(f.value_clean), 3)  AS daily_avg,
  ROUND(MAX(f.value_clean), 3)  AS daily_max,
  ROUND(MIN(f.value_clean), 3)  AS daily_min,
  COUNT(*)                      AS valid_hours,
  ROUND(100.0 * COUNT(*) / 24, 1) AS completeness_pct
FROM workspace.aq_silver.fact_observation f
JOIN workspace.aq_silver.dim_station s USING (site_id)
GROUP BY 1,2,3,4,5,6,7,8
""")

spark.sql("SELECT count(*) AS rows FROM workspace.aq_gold.daily_site_summary").show()

+------+
|  rows|
+------+
|152176|
+------+



## 2. Hourly profile

**Question it answers:** what does a typical day look like at this station?

`PERCENTILE_APPROX` rather than an exact percentile: exact percentiles require
sorting every partition, which is expensive on 3.5M rows. The approximation is
accurate enough for a profile chart.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.site_hourly_profile AS
SELECT
  f.site_id,
  s.site_name,
  s.region,
  f.parameter_code,
  f.obs_hour,
  ROUND(AVG(f.value_clean), 3)                     AS mean_value,
  ROUND(PERCENTILE_APPROX(f.value_clean, 0.5), 3)  AS median_value,
  ROUND(PERCENTILE_APPROX(f.value_clean, 0.95), 3) AS p95_value,
  COUNT(*)                                         AS readings
FROM workspace.aq_silver.fact_observation f
JOIN workspace.aq_silver.dim_station s USING (site_id)
GROUP BY 1,2,3,4,5
""")

spark.sql("""
SELECT site_name, parameter_code, obs_hour, mean_value
FROM workspace.aq_gold.site_hourly_profile
WHERE parameter_code = 'NO2'
ORDER BY site_name, obs_hour
""").show(24)

+----------+--------------+--------+----------+
| site_name|parameter_code|obs_hour|mean_value|
+----------+--------------+--------+----------+
|Alexandria|           NO2|       0|     0.902|
|Alexandria|           NO2|       1|     0.707|
|Alexandria|           NO2|       2|     0.809|
|Alexandria|           NO2|       3|     0.837|
|Alexandria|           NO2|       4|     0.931|
|Alexandria|           NO2|       5|     1.153|
|Alexandria|           NO2|       6|     1.347|
|Alexandria|           NO2|       7|     1.319|
|Alexandria|           NO2|       8|     1.134|
|Alexandria|           NO2|       9|     0.938|
|Alexandria|           NO2|      10|     0.786|
|Alexandria|           NO2|      11|      0.67|
|Alexandria|           NO2|      12|     0.579|
|Alexandria|           NO2|      13|     0.533|
|Alexandria|           NO2|      14|     0.508|
|Alexandria|           NO2|      15|     0.526|
|Alexandria|           NO2|      16|     0.627|
|Alexandria|           NO2|      17|    

### CHECKPOINT — a free correctness check

NO2 comes largely from vehicle exhaust, so a roadside station should show **two
humps** in the output above: a morning peak and an evening peak.

| What you see | Meaning |
|---|---|
| Peaks around hours 7–9 and 17–19 | Timestamp arithmetic is correct |
| Peaks around hours 1–3 | The 1-based hour bug is still present. Go back to silver |

Domain knowledge catching a bug that no amount of code review would.

## 3. Regional trend

**Question it answers:** is air quality getting better or worse?

This is where the window functions live — what job ads mean by "advanced SQL".

| Window clause | Meaning |
|---|---|
| `PARTITION BY region, parameter_code` | Restart the calculation per region-pollutant |
| `ORDER BY month_start` | The sequence within that group |
| `ROWS BETWEEN 11 PRECEDING AND CURRENT ROW` | This month plus the 11 before it |

`LAG(monthly_avg, 12)` reaches back exactly 12 rows — the same month a year
earlier — giving a seasonally fair year-on-year comparison.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.regional_trend AS
WITH monthly AS (
  SELECT
    s.region,
    f.parameter_code,
    DATE_TRUNC('MONTH', f.obs_date) AS month_start,
    ROUND(AVG(f.value_clean), 3)    AS monthly_avg,
    COUNT(DISTINCT f.site_id)       AS stations,
    COUNT(*)                        AS readings
  FROM workspace.aq_silver.fact_observation f
  JOIN workspace.aq_silver.dim_station s USING (site_id)
  GROUP BY 1, 2, 3
)
SELECT
  *,
  ROUND(AVG(monthly_avg) OVER (
    PARTITION BY region, parameter_code ORDER BY month_start
    ROWS BETWEEN 11 PRECEDING AND CURRENT ROW), 3) AS rolling_12m,
  ROUND(LAG(monthly_avg, 12) OVER (
    PARTITION BY region, parameter_code ORDER BY month_start), 3) AS same_month_last_year,
  ROUND(100.0 * (monthly_avg - LAG(monthly_avg, 12) OVER (
    PARTITION BY region, parameter_code ORDER BY month_start))
    / NULLIF(LAG(monthly_avg, 12) OVER (
        PARTITION BY region, parameter_code ORDER BY month_start), 0), 1) AS yoy_pct_change
FROM monthly
""")

spark.sql("""
SELECT region, month_start, monthly_avg, rolling_12m, yoy_pct_change
FROM workspace.aq_gold.regional_trend
WHERE parameter_code = 'PM2.5'
ORDER BY month_start
LIMIT 15
""").show()

+-----------------+-------------------+-----------+-----------+--------------+
|           region|        month_start|monthly_avg|rolling_12m|yoy_pct_change|
+-----------------+-------------------+-----------+-----------+--------------+
|      Sydney East|2020-01-01 00:00:00|     20.235|     20.235|          NULL|
|Sydney North-west|2020-01-01 00:00:00|     21.097|     21.097|          NULL|
|Sydney South-west|2020-01-01 00:00:00|      25.36|      25.36|          NULL|
|Sydney South-west|2020-02-01 00:00:00|      6.863|     16.112|          NULL|
|Sydney North-west|2020-02-01 00:00:00|      6.192|     13.645|          NULL|
|      Sydney East|2020-02-01 00:00:00|      6.278|     13.256|          NULL|
|Sydney South-west|2020-03-01 00:00:00|      5.667|      12.63|          NULL|
|      Sydney East|2020-03-01 00:00:00|      4.932|     10.482|          NULL|
|Sydney North-west|2020-03-01 00:00:00|      4.959|     10.749|          NULL|
|Sydney South-west|2020-04-01 00:00:00|      7.687| 

### What to look for

January 2020 should show a dramatic PM2.5 spike. That is the Black Summer
bushfire smoke — your first sight of the event you formally detect at the weekend.

## 4. Exceedance events

**Question it answers:** how often is the air actually unhealthy?

`WHERE completeness_pct >= 75` — a "daily average" from 4 hours of data should not
be compared against a 24-hour standard. Filtering it out is more honest than
including it.

**Caveat to record in the README:** the thresholds below are approximate
NEPM-style values used for illustration. Look them up and cite the source if you
want them exact. Never present made-up thresholds as regulatory fact.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.aq_gold.exceedance_events AS
WITH standards AS (
  SELECT * FROM VALUES
    ('PM2.5', 25.0),
    ('PM10',  50.0)
  AS t(parameter_code, daily_standard)
),
flagged AS (
  SELECT
    d.site_id, d.site_name, d.region, d.obs_date, d.parameter_code,
    d.daily_avg, s.daily_standard, d.completeness_pct,
    CASE WHEN d.daily_avg > s.daily_standard THEN 1 ELSE 0 END AS is_exceedance
  FROM workspace.aq_gold.daily_site_summary d
  JOIN standards s USING (parameter_code)
  WHERE d.completeness_pct >= 75
),
runs AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY site_id, parameter_code ORDER BY obs_date)
      - ROW_NUMBER() OVER (PARTITION BY site_id, parameter_code, is_exceedance ORDER BY obs_date)
      AS run_group
  FROM flagged
)
SELECT
  site_id, site_name, region, obs_date, parameter_code,
  daily_avg, daily_standard,
  ROUND(daily_avg / daily_standard, 2) AS times_standard,
  COUNT(*) OVER (PARTITION BY site_id, parameter_code, run_group) AS consecutive_days
FROM runs
WHERE is_exceedance = 1
""")

spark.sql("""
SELECT parameter_code,
       count(*) AS exceedance_days,
       max(consecutive_days) AS longest_run,
       min(obs_date) AS first_date,
       max(obs_date) AS last_date
FROM workspace.aq_gold.exceedance_events
GROUP BY 1 ORDER BY 2 DESC
""").show()

+--------------+---------------+-----------+----------+----------+
|parameter_code|exceedance_days|longest_run|first_date| last_date|
+--------------+---------------+-----------+----------+----------+
|         PM2.5|            272|          5|2020-01-01|2025-12-09|
|          PM10|            186|          5|2020-01-01|2025-12-21|
+--------------+---------------+-----------+----------+----------+



### The gaps-and-islands pattern

`consecutive_days` uses two row numbers over the same rows — one ordered by date,
one ordered by date within each exceedance flag. Their difference is constant
across an unbroken run:

| date | is_exceedance | rn_all | rn_by_flag | difference |
|---|---|---|---|---|
| Jan 1 | 1 | 1 | 1 | 0 |
| Jan 2 | 1 | 2 | 2 | 0 |
| Jan 3 | 0 | 3 | 1 | 2 |
| Jan 4 | 1 | 4 | 3 | 1 |

Rows sharing a difference form one consecutive run. You will use this identical
trick at the weekend for flatline detection.

## 5. Prove every grain

Every line must print `0`. If one does not, that table's grain is not what the
documentation claims, and every number built on it is unreliable.

In [0]:
checks = [
    ("daily_site_summary",  "site_id, obs_date, parameter_code"),
    ("site_hourly_profile", "site_id, parameter_code, obs_hour"),
    ("regional_trend",      "region, parameter_code, month_start"),
    ("exceedance_events",   "site_id, obs_date, parameter_code"),
]

for table, keys in checks:
    n = spark.sql(f"""
        SELECT {keys}, count(*) AS c
        FROM workspace.aq_gold.{table}
        GROUP BY {keys} HAVING count(*) > 1
    """).count()
    print(f"{table:22} grain violations: {n}")

daily_site_summary     grain violations: 0
site_hourly_profile    grain violations: 0
regional_trend         grain violations: 0
exceedance_events      grain violations: 0


## 6. Table comments — feeds Genie

Genie writes SQL from plain-English questions. Its quality depends almost entirely
on this metadata. Without comments it guesses; with them it reads.

In [0]:
spark.sql("""
COMMENT ON TABLE workspace.aq_gold.daily_site_summary IS
  'One row per monitoring station, per date, per pollutant. Daily average, maximum, minimum, and the percentage of the 24 hours that had valid readings.'
""")

spark.sql("""
ALTER TABLE workspace.aq_gold.daily_site_summary
  ALTER COLUMN daily_avg COMMENT 'Mean hourly concentration for the day, in the parameter native units'
""")

spark.sql("""
ALTER TABLE workspace.aq_gold.daily_site_summary
  ALTER COLUMN completeness_pct COMMENT 'Percentage of the 24 hours that produced a valid reading. Below 75 percent the daily average is unreliable'
""")

spark.sql("""
COMMENT ON TABLE workspace.aq_gold.regional_trend IS
  'One row per region, per month, per pollutant. Monthly average with a 12-month rolling mean and year-on-year percentage change.'
""")

spark.sql("""
COMMENT ON TABLE workspace.aq_gold.site_hourly_profile IS
  'One row per station, per pollutant, per hour of day. The typical daily pollution curve across the whole period.'
""")

spark.sql("""
COMMENT ON TABLE workspace.aq_gold.exceedance_events IS
  'One row per station-day where the daily average exceeded the air quality standard for that pollutant. Includes the length of the consecutive exceedance run.'
""")

print("comments applied")

comments applied


In [0]:
spark.sql("""
SELECT parameter_code,
       count(*) AS negatives,
       round(min(value), 3) AS most_negative,
       round(percentile_approx(value, 0.5), 3) AS median_negative
FROM workspace.aq_quarantine.observations
WHERE reject_reason = 'negative_value'
GROUP BY 1 ORDER BY 2 DESC
""").show()

+--------------+---------+-------------+---------------+
|parameter_code|negatives|most_negative|median_negative|
+--------------+---------+-------------+---------------+
+--------------+---------+-------------+---------------+



In [0]:
spark.sql("""
SELECT parameter_code,
       count(*) AS at_or_near_floor
FROM workspace.aq_quarantine.observations
WHERE reject_reason = 'negative_value' AND value <= -9.9
GROUP BY 1 ORDER BY 2 DESC
""").show()

+--------------+----------------+
|parameter_code|at_or_near_floor|
+--------------+----------------+
+--------------+----------------+



In [0]:
spark.sql("""
SELECT parameter_code,
       ROUND(percentile_approx(value, 0.01), 2) AS p01,
       ROUND(percentile_approx(value, 0.10), 2) AS p10,
       ROUND(percentile_approx(value, 0.25), 2) AS p25,
       ROUND(percentile_approx(value, 0.50), 2) AS p50,
       ROUND(percentile_approx(value, 0.90), 2) AS p90
FROM workspace.aq_quarantine.observations
WHERE reject_reason = 'negative_value'
GROUP BY 1 ORDER BY 1
""").show()

+--------------+---+---+---+---+---+
|parameter_code|p01|p10|p25|p50|p90|
+--------------+---+---+---+---+---+
+--------------+---+---+---+---+---+



In [0]:
spark.sql("""
WITH floored AS (
  SELECT parameter_code, ROUND(AVG(GREATEST(value, 0)), 3) AS avg_floored
  FROM (
    SELECT parameter_code, value FROM workspace.aq_silver.fact_observation
    UNION ALL
    SELECT parameter_code, value FROM workspace.aq_quarantine.observations
    WHERE reject_reason = 'negative_value'
  ) GROUP BY 1
),
excluded AS (
  SELECT parameter_code, ROUND(AVG(value), 3) AS avg_excluded
  FROM workspace.aq_silver.fact_observation GROUP BY 1
)
SELECT e.parameter_code, avg_excluded, avg_floored,
       ROUND(100.0 * (avg_excluded - avg_floored) / avg_floored, 2) AS pct_overstated
FROM excluded e JOIN floored f USING (parameter_code)
ORDER BY 4 DESC
""").show()

+--------------+------------+-----------+--------------+
|parameter_code|avg_excluded|avg_floored|pct_overstated|
+--------------+------------+-----------+--------------+
|         OZONE|       1.755|      1.756|         -0.06|
|          PM10|      15.356|     15.392|         -0.23|
|           NO2|       0.636|      0.641|         -0.78|
|         PM2.5|       6.324|      6.538|         -3.27|
+--------------+------------+-----------+--------------+

